# Análise de Crescimento Populacional Brasil (2010-2022)

Este notebook realiza uma análise completa dos dados de população do Brasil comparando os dados do Censo 2010 com o Censo 2022, agregando a informação por estado e por município.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

diretorio = r'H:\Py3etapa\At4'
arquivo_fonte = os.path.join(diretorio, 'CD2022_Populacao_2010_Compatibilizada_20231222.xlsx')

print("✓ Bibliotecas importadas com sucesso")
print(f"✓ Diretório de trabalho: {diretorio}")

## 1. Carregamento e Limpeza dos Dados

In [ ]:
df = pd.read_excel(arquivo_fonte, header=2)

df = df.dropna(subset=['UF'], how='all')
df = df[df['UF'].notna()].copy()
df = df[df['UF'] != 'UF'].copy()

print(f"✓ Dados carregados: {len(df)} linhas")
print(f"\nDimensões: {df.shape}")
print(f"\nPrimeiras linhas:")
print(df.head())

In [ ]:
col_2010 = 'População Município 2010\n(Sinopse)'
col_2010_alt = 'População 2010 (Alterações de Limites até 2022)1'
col_2022 = 'População Censo 2022'

df[col_2010] = pd.to_numeric(df[col_2010], errors='coerce')
df[col_2010_alt] = pd.to_numeric(df[col_2010_alt], errors='coerce')
df[col_2022] = pd.to_numeric(df[col_2022], errors='coerce')

print("✓ Colunas numéricas convertidas com sucesso")
print(f"\nEstatísticas descritivas:")
print(df[[col_2010, col_2010_alt, col_2022]].describe())

## 2. Agregação por Estado

In [ ]:
df_estado = df.groupby('UF').agg({
    col_2010: 'sum',
    col_2010_alt: 'sum',
    col_2022: 'sum'
}).reset_index()

df_estado['Crescimento (2022 - 2010)'] = df_estado[col_2022] - df_estado[col_2010_alt]

df_estado['Taxa de Crescimento (%)'] = (df_estado['Crescimento (2022 - 2010)'] / df_estado[col_2010_alt] * 100).round(2)

df_estado_ordenado = df_estado.sort_values('Crescimento (2022 - 2010)', ascending=False).reset_index(drop=True)

print(f"✓ Total de estados: {len(df_estado_ordenado)}")
print(f"\nEstados ordenados por crescimento (2022 - 2010):")
print(df_estado_ordenado[['UF', col_2010_alt, col_2022, 'Crescimento (2022 - 2010)', 'Taxa de Crescimento (%)']].to_string())

In [ ]:
arquivo_estado_csv = os.path.join(diretorio, 'populacao_por_estado.csv')
df_estado_ordenado.to_csv(arquivo_estado_csv, sep=';', index=False, encoding='utf-8')
print(f"✓ Arquivo salvo: populacao_por_estado.csv")

print(f"\nTop 10 Estados com Maior Crescimento:")
print(df_estado_ordenado[['UF', 'Crescimento (2022 - 2010)', 'Taxa de Crescimento (%)']].head(10).to_string())

## 3. Agregação por Município

In [ ]:
df_municipio = df.groupby(['UF', 'NOME DO MUNICÍPIO']).agg({
    col_2010: 'sum',
    col_2010_alt: 'sum',
    col_2022: 'sum'
}).reset_index()

df_municipio['Crescimento (2022 - 2010)'] = df_municipio[col_2022] - df_municipio[col_2010_alt]

df_municipio['Taxa de Crescimento (%)'] = (df_municipio['Crescimento (2022 - 2010)'] / df_municipio[col_2010_alt] * 100).round(2)

df_municipio_ordenado = df_municipio.sort_values('Crescimento (2022 - 2010)', ascending=False).reset_index(drop=True)

print(f"✓ Total de municípios: {len(df_municipio_ordenado)}")
print(f"\nTop 30 Municípios com Maior Crescimento Absoluto:")
top_30 = df_municipio_ordenado[['UF', 'NOME DO MUNICÍPIO', col_2010_alt, col_2022, 'Crescimento (2022 - 2010)', 'Taxa de Crescimento (%)']].head(30)
print(top_30.to_string())

In [ ]:
arquivo_municipio_csv = os.path.join(diretorio, 'populacao_por_municipio.csv')
df_municipio_ordenado.to_csv(arquivo_municipio_csv, sep=';', index=False, encoding='utf-8')
print(f"✓ Arquivo salvo: populacao_por_municipio.csv")

print(f"\nEstatísticas dos Municípios:")
print(f"  Total de municípios: {len(df_municipio_ordenado)}")
print(f"  Crescimento mínimo: {df_municipio_ordenado['Crescimento (2022 - 2010)'].min():.0f} hab.")
print(f"  Crescimento máximo: {df_municipio_ordenado['Crescimento (2022 - 2010)'].max():.0f} hab.")
print(f"  Crescimento médio: {df_municipio_ordenado['Crescimento (2022 - 2010)'].mean():.0f} hab.")

## 4. Análise de Municípios com Maior Taxa de Crescimento

In [ ]:
df_municipio_pop_min = df_municipio_ordenado[df_municipio_ordenado[col_2010_alt] >= 10000].copy()

print(f"Municípios com população >= 10.000 hab em 2010: {len(df_municipio_pop_min)}")
print(f"\nTop 20 Municípios (Pop. >= 10k) por Taxa de Crescimento (%):")
top_taxa = df_municipio_pop_min.sort_values('Taxa de Crescimento (%)', ascending=False).head(20)
print(top_taxa[['UF', 'NOME DO MUNICÍPIO', col_2010_alt, col_2022, 'Taxa de Crescimento (%)']].to_string())

## 5. Visualizações

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
top_15_estado = df_estado_ordenado.head(15)
ax.barh(top_15_estado['UF'], top_15_estado['Crescimento (2022 - 2010)'], color='steelblue')
ax.set_xlabel('Crescimento Populacional (2022 - 2010)', fontsize=12)
ax.set_ylabel('Estado', fontsize=12)
ax.set_title('Top 15 Estados com Maior Crescimento Populacional (2010-2022)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(diretorio, 'top_15_estados_crescimento.png'), dpi=100, bbox_inches='tight')
plt.show()
print("✓ Gráfico salvo: top_15_estados_crescimento.png")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
top_15_municipio = df_municipio_ordenado.head(15)
ax.barh(top_15_municipio['NOME DO MUNICÍPIO'], top_15_municipio['Crescimento (2022 - 2010)'], color='coral')
ax.set_xlabel('Crescimento Populacional (2022 - 2010)', fontsize=12)
ax.set_ylabel('Município', fontsize=12)
ax.set_title('Top 15 Municípios com Maior Crescimento Populacional (2010-2022)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(diretorio, 'top_15_municipios_crescimento.png'), dpi=100, bbox_inches='tight')
plt.show()
print("✓ Gráfico salvo: top_15_municipios_crescimento.png")

## 6. Resumo Final

In [ ]:
print("="*80)
print("RESUMO EXECUTIVO - ANÁLISE DE CRESCIMENTO POPULACIONAL (2010-2022)")
print("="*80)

pop_2010_total = df[col_2010_alt].sum()
pop_2022_total = df[col_2022].sum()
crescimento_total = pop_2022_total - pop_2010_total
taxa_crescimento_total = (crescimento_total / pop_2010_total * 100)

print(f"\n📊 BRASIL - TOTAIS:")
print(f"  População 2010: {pop_2010_total:,.0f} habitantes")
print(f"  População 2022: {pop_2022_total:,.0f} habitantes")
print(f"  Crescimento: {crescimento_total:,.0f} habitantes")
print(f"  Taxa de Crescimento: {taxa_crescimento_total:.2f}%")

print(f"\n🏆 ESTADOS COM MAIOR CRESCIMENTO:")
for i, row in df_estado_ordenado.head(5).iterrows():
    print(f"  {i+1}. {row['UF']}: {row['Crescimento (2022 - 2010)']:,.0f} hab. ({row['Taxa de Crescimento (%)']:.2f}%)")

print(f"\n🏙️  MUNICÍPIOS COM MAIOR CRESCIMENTO:")
for i, row in df_municipio_ordenado.head(5).iterrows():
    print(f"  {i+1}. {row['NOME DO MUNICÍPIO']} ({row['UF']}): {row['Crescimento (2022 - 2010)']:,.0f} hab. ({row['Taxa de Crescimento (%)']:.2f}%)")

print(f"\n📁 ARQUIVOS GERADOS:")
print(f"  1. populacao_por_estado.csv")
print(f"  2. populacao_por_municipio.csv")
print(f"  3. top_15_estados_crescimento.png")
print(f"  4. top_15_municipios_crescimento.png")
print(f"\n✓ Todos os arquivos salvos em: H:\\Py3etapa\\At4\\")
print("="*80)